# py-web-audio-api notebook quickstart

This notebook shows two good Jupyter paths:

- `OfflineAudioContext` for inline playback that works well in notebooks.
- `AudioContext` for live local playback when the kernel is running on your own machine and can access your host audio device.

## Before opening this notebook

Install the latest published PyPI release into the same environment that Jupyter will use as its kernel:

```bash
python3 -m venv .notebook
.notebook/bin/python -m pip install --upgrade pip
.notebook/bin/python -m pip install web-audio-api jupyter ipykernel matplotlib ipympl
.notebook/bin/python -m ipykernel install --user --name py-web-audio-api
.notebook/bin/jupyter lab
```

`ipykernel install` only registers a kernel. It does not install `web_audio_api`, so make sure the `py-web-audio-api` kernel points at the same environment where you installed `web-audio-api`.

Then pick the `py-web-audio-api` kernel in Jupyter.

In [ ]:
import asyncio
import io
import math

import ipywidgets as widgets
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
import web_audio_api
from IPython.display import Audio, display


Jupyter already runs an event loop, so use top-level `await` instead of `asyncio.run(...)`.

## Offline render for inline notebook playback

In [ ]:
sample_rate = 44_100.0
duration = 1.5
length = int(sample_rate * duration)

ctx = web_audio_api.OfflineAudioContext(1, length, sample_rate)

osc = ctx.createOscillator()
gain = ctx.createGain()

osc.type = "sine"
osc.frequency.value = 220.0
gain.gain.value = 0.15

osc.connect(gain)
gain.connect(ctx.destination)

osc.start(0.0)
osc.stop(duration)

rendered = await ctx.startRendering()
samples = rendered.getChannelData(0)

len(samples), min(samples), max(samples)

In [ ]:
Audio(samples, rate=int(sample_rate))

You can also build buffers directly in Python and play the rendered result inline.

In [ ]:
sample_rate = 44_100.0
duration = 1.0
length = int(sample_rate * duration)

ctx = web_audio_api.OfflineAudioContext(1, length, sample_rate)
buffer = ctx.createBuffer(1, length, sample_rate)

samples = [
    math.sin(2.0 * math.pi * 330.0 * i / sample_rate)
    * (1.0 - (i / length))
    * 0.2
    for i in range(length)
]
buffer.copyToChannel(samples, 0)

src = ctx.createBufferSource()
src.buffer = buffer
src.connect(ctx.destination)
src.start(0.0)

rendered = await ctx.startRendering()
Audio(rendered.getChannelData(0), rate=int(sample_rate))

## Live local playback

This section is for local kernels. If your notebook kernel is running on your machine, `AudioContext()` can use the host audio device for realtime playback.

In [ ]:
if "live_monitor_task" in globals():
    live_monitor_task.cancel()
    try:
        await live_monitor_task
    except asyncio.CancelledError:
        pass

if "live_ctx" in globals():
    await live_ctx.close()

live_ctx = web_audio_api.AudioContext()
live_osc = live_ctx.createOscillator()
live_gain = live_ctx.createGain()
live_analyser = live_ctx.createAnalyser()

live_osc.type = "sawtooth"
live_osc.frequency.value = 110.0
live_gain.gain.value = 0.05

live_analyser.fftSize = 2048
live_analyser.smoothingTimeConstant = 0.85

live_osc.connect(live_gain)
live_gain.connect(live_ctx.destination)
live_gain.connect(live_analyser)

await live_ctx.resume()
live_osc.start()
"Playing a live sawtooth oscillator at 110 Hz. Run the next cell to start a persistent live analyser monitor."

In [ ]:
async def run_live_analyser(ctx, analyser, image, max_hz=2_000.0):
    def as_float_list(values):
        return [float(value) for value in values]

    fft_size = analyser.fftSize
    bin_count = analyser.frequencyBinCount
    sample_rate = ctx.sampleRate

    time_axis_ms = [1000.0 * i / sample_rate for i in range(fft_size)]
    freq_axis_hz = [i * sample_rate / fft_size for i in range(bin_count)]

    fig = Figure(figsize=(10, 6))
    FigureCanvasAgg(fig)
    ax_time = fig.add_subplot(2, 1, 1)
    ax_freq = fig.add_subplot(2, 1, 2)
    wave_line, = ax_time.plot(time_axis_ms, [0.0] * fft_size, color="#14532d", linewidth=1.5)
    freq_line, = ax_freq.plot(freq_axis_hz, [0.0] * bin_count, color="#b45309", linewidth=1.5)

    ax_time.set_title("Live waveform")
    ax_time.set_xlabel("Time (ms)")
    ax_time.set_ylabel("Amplitude")
    ax_time.set_ylim(-1.1, 1.1)
    ax_time.grid(alpha=0.25)

    ax_freq.set_title("Live frequency spectrum")
    ax_freq.set_xlabel("Frequency (Hz)")
    ax_freq.set_ylabel("Magnitude")
    ax_freq.set_xlim(0.0, max_hz)
    ax_freq.set_ylim(0, 255)
    ax_freq.grid(alpha=0.25)
    fig.tight_layout()

    try:
        while True:
            waveform = as_float_list(analyser.getFloatTimeDomainData([0.0] * fft_size))
            spectrum = as_float_list(analyser.getByteFrequencyData([0] * bin_count))
            peak = max((abs(value) for value in waveform), default=0.0)
            ylim = min(1.1, max(0.04, peak * 1.8))
            wave_line.set_ydata(waveform)
            ax_time.set_ylim(-ylim, ylim)
            freq_line.set_ydata(spectrum)

            buffer = io.BytesIO()
            fig.savefig(buffer, format="png", dpi=110)
            image.value = buffer.getvalue()
            await asyncio.sleep(0.08)
    finally:
        fig.clear()


def start_live_analyser(ctx, analyser, max_hz=2_000.0):
    global live_monitor_image
    if "live_monitor_image" not in globals():
        live_monitor_image = widgets.Image(format="png")
        display(live_monitor_image)
    image = live_monitor_image
    return asyncio.create_task(run_live_analyser(ctx, analyser, image, max_hz=max_hz))


if "live_monitor_task" in globals():
    live_monitor_task.cancel()
    try:
        await live_monitor_task
    except asyncio.CancelledError:
        pass
    if "live_monitor_image" in globals():
        live_monitor_image.value = b""

live_monitor_task = start_live_analyser(live_ctx, live_analyser)
"Live analyser monitor started. It will keep updating while you run later cells."

In [ ]:
live_osc.frequency.value = 220.0
live_osc.type = "square"
live_gain.gain.value = 0.03

"The live graph should keep updating while the pitch rises and the waveform changes shape."

In [ ]:
if "live_monitor_task" in globals():
    live_monitor_task.cancel()
    try:
        await live_monitor_task
    except asyncio.CancelledError:
        pass

live_osc.stop()
await live_ctx.close()
del live_ctx, live_osc, live_gain, live_analyser, live_monitor_task

"Live audio stopped and context closed."